# TDNet Sunday/Monday results review

Run this notebook after the latest games are final. It scores the immutable prediction bundle against the newest results, then displays the prediction-vs-results table, model scorecard, and the review figure. It never edits prediction bytes or posts externally.

In [ ]:
from pathlib import Path
import json
import subprocess
import shutil
import sys
import pandas as pd
from IPython.display import Image, display
PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'src' / 'gridiron_ml').exists())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))


## Select the completed week

Set `WEEK` to the most recent completed prediction bundle.

In [ ]:
SEASON = 2026
WEEK = 1
PREDICTION_BUNDLE = PROJECT_ROOT / 'data' / 'publication' / str(SEASON) / 'weekly_predictions' / f'week_{WEEK:02d}'
RESULTS = PROJECT_ROOT / 'data' / 'raw' / 'cfbd' / 'v2' / 'games' / f'{SEASON}.parquet'
SNAPSHOT = PROJECT_ROOT / 'data' / 'publication' / str(SEASON) / 'weekly_operations' / 'snapshot_completeness.json'
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'publication' / str(SEASON) / 'sunday_review' / f'week_{WEEK:02d}'
FIGURE_ROOT = PROJECT_ROOT / 'publication' / str(SEASON) / 'figures' / 'sunday_review' / f'week_{WEEK:02d}'
assert PREDICTION_BUNDLE.exists(), f'Missing immutable prediction bundle: {PREDICTION_BUNDLE}'
assert RESULTS.exists(), f'Missing completed results: {RESULTS}'
assert SNAPSHOT.exists(), f'Missing certified snapshot report: {SNAPSHOT}'
print(PREDICTION_BUNDLE)


In [ ]:
command = [sys.executable, str(PROJECT_ROOT / 'src/gridiron_ml/cli/publication/run_sunday_publication_pipeline.py'), '--bundle', str(PREDICTION_BUNDLE), '--results', str(RESULTS), '--snapshot-completeness', str(SNAPSHOT), '--output-root', str(OUTPUT_ROOT)]
subprocess.run(command, cwd=PROJECT_ROOT, check=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copy2(OUTPUT_ROOT / 'sunday_performance.png', FIGURE_ROOT / 'sunday_performance.png')
shutil.copy2(OUTPUT_ROOT / 'sunday_performance.svg', FIGURE_ROOT / 'sunday_performance.svg')
scorecard = pd.read_csv(OUTPUT_ROOT / 'scorecard.csv')
scored = pd.read_parquet(OUTPUT_ROOT / 'scoring' / 'scored_predictions.parquet')
display(scorecard)


## Prediction versus results

In [ ]:
columns = [column for column in ['week', 'game_id', 'home_team', 'away_team', 'pred_home_margin', 'actual_margin', 'absolute_margin_error', 'pred_home_win_probability', 'actual_home_win', 'winner_correct'] if column in scored]
display(scored[columns].sort_values(['week', 'game_id']))
display(Image(filename=str(OUTPUT_ROOT / 'sunday_performance.png')))
